# Orthogonal Projections & QR Factorization
## From Inner Products to Least Squares — A From-Scratch Exploration

---

**Author:** Computational Mathematics Notebook Series  
**Topic:** Gram-Schmidt, QR Factorization, Least Squares, Function Approximation  
**Prerequisites:** Linear algebra (vector spaces, matrix operations), Python/NumPy  
**Primary Reference:** Trefethen, L. N., & Bau, D. (1997). *Numerical Linear Algebra*. SIAM.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import linalg as la
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Tolerances ─────────────────────────────────────────────────────────────────
ATOL = 1e-10
RTOL = 1e-6

# ── Plot Style ─────────────────────────────────────────────────────────────────
PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'mediumpurple'
GRAY      = '#888888'

plt.rcParams.update({
    'figure.figsize'  : (12, 5),
    'font.size'       : 12,
    'axes.grid'       : True,
    'grid.alpha'      : 0.3,
    'lines.linewidth' : 2,
    'axes.spines.top' : False,
    'axes.spines.right': False,
})

def check(condition, label):
    status = 'PASS' if condition else 'FAIL'
    print(f'  [{status}]  {label}')
    return condition

print('Setup complete.')

---
## 1.  Problem Statement

### 1.1  Why Orthogonality?

Orthogonality is one of the most powerful organizing principles in linear algebra and numerical computation. When a set of vectors is mutually orthogonal, computations simplify dramatically: projections become independent, least-squares solutions reduce to back-substitution, and the normal equations become diagonal.

The core question: **given a subspace $\mathcal{S}$ and a vector $b$, what point in $\mathcal{S}$ is closest to $b$?** The answer is always the orthogonal projection of $b$ onto $\mathcal{S}$.

### 1.2  Applications

| Domain | Application | Why Orthogonality Helps |
|--------|------------|-------------------------|
| Data fitting | Linear regression ($Ax \approx b$) | QR solves least squares stably |
| Signal processing | Fourier analysis | Fourier basis is orthonormal in $L^2$ |
| Machine learning | PCA / feature extraction | Orthogonal principal components |
| Numerical ODE | Krylov subspace methods | Arnoldi / Lanczos build ONBs iteratively |
| Statistics | ANOVA / regression | Orthogonal contrasts decompose variance |
| Function spaces | Polynomial/spectral approximation | Orthogonal polynomial bases (Legendre, Chebyshev) |

### 1.3  Road Map

```
Inner Product Spaces
       ↓
Orthogonal Projection  →  Projection Matrix P = A(AᵀA)⁻¹Aᵀ
       ↓
Gram-Schmidt (CGS, MGS)  →  QR Factorization A = QR
       ↓
Least Squares via QR  →  x = R⁻¹Qᵀb
       ↓
Function Approximation in L²[-π,π]
```

---
## 2.  Inner Product Spaces

### 2.1  Definition

An **inner product** on a vector space $V$ over $\mathbb{R}$ is a map $\langle \cdot, \cdot \rangle : V \times V \to \mathbb{R}$ satisfying:

1. **Symmetry:** $\langle u, v \rangle = \langle v, u \rangle$
2. **Linearity in first argument:** $\langle \alpha u + \beta w, v \rangle = \alpha\langle u, v\rangle + \beta\langle w, v\rangle$
3. **Positive definiteness:** $\langle v, v \rangle \geq 0$, with equality iff $v = 0$

The induced **norm** is $\|v\| = \sqrt{\langle v, v \rangle}$.

### 2.2  Examples

**Euclidean space $\mathbb{R}^n$:**
$$\langle u, v \rangle = u^\top v = \sum_{i=1}^n u_i v_i$$

**Weighted inner product** (used in statistics/FEM):
$$\langle u, v \rangle_W = u^\top W v, \quad W \succ 0$$

**Function space $L^2[a,b]$:**
$$\langle f, g \rangle = \int_a^b f(x)\, g(x)\, dx$$

Two vectors are **orthogonal** if $\langle u, v \rangle = 0$, written $u \perp v$.

### 2.3  Cauchy-Schwarz Inequality

For any inner product space:
$$\boxed{|\langle u, v \rangle| \leq \|u\| \cdot \|v\|}$$

with equality iff $u$ and $v$ are linearly dependent. This defines the **angle** between vectors:
$$\cos\theta = \frac{\langle u, v \rangle}{\|u\|\,\|v\|}$$

**Proof sketch:** Consider $\|u - t v\|^2 \geq 0$ for all $t \in \mathbb{R}$. Expanding and minimizing over $t = \langle u,v\rangle / \|v\|^2$ gives Cauchy-Schwarz. $\square$

In [ ]:
# =============================================================================
# Section 2 — Inner Product Verification
# =============================================================================

def euclidean_inner(u, v):
    """Standard Euclidean inner product.

    Args:
        u: First vector.   Shape: (n,)
        v: Second vector.  Shape: (n,)

    Returns:
        Scalar inner product <u, v>.
    """
    return float(np.dot(u, v))

def cauchy_schwarz_demo(n=8):
    """Demonstrate the Cauchy-Schwarz inequality on random vectors."""
    u = rng.standard_normal(n)
    v = rng.standard_normal(n)

    lhs = abs(euclidean_inner(u, v))
    rhs = np.linalg.norm(u) * np.linalg.norm(v)
    cos_theta = lhs / rhs

    print('Cauchy-Schwarz Demonstration')
    print(f'  |<u, v>|        = {lhs:.6f}')
    print(f'  ||u|| * ||v||   = {rhs:.6f}')
    print(f'  cos(theta)      = {cos_theta:.6f}')
    check(lhs <= rhs + 1e-14, f'|<u,v>| <= ||u|| ||v||  ({lhs:.4f} <= {rhs:.4f})')

    # Equality case: v parallel to u
    v_parallel = 3.7 * u
    lhs_eq = abs(euclidean_inner(u, v_parallel))
    rhs_eq = np.linalg.norm(u) * np.linalg.norm(v_parallel)
    check(abs(lhs_eq - rhs_eq) < 1e-10, f'Equality for parallel vectors: {lhs_eq:.6f} == {rhs_eq:.6f}')

cauchy_schwarz_demo()

---
## 3.  Orthogonal Projection Theory

### 3.1  Projection onto a Subspace

Let $\mathcal{S} = \mathrm{col}(A)$ for $A \in \mathbb{R}^{m \times n}$ with $\mathrm{rank}(A) = n$. The **orthogonal projection** of $b \in \mathbb{R}^m$ onto $\mathcal{S}$ is the unique vector $\hat{b} \in \mathcal{S}$ minimizing $\|b - \hat{b}\|$.

**Optimality condition:** The error $e = b - \hat{b}$ must be orthogonal to every vector in $\mathcal{S}$:
$$A^\top (b - A\hat{x}) = 0 \implies A^\top A \hat{x} = A^\top b$$

These are the **normal equations**. Solving: $\hat{x} = (A^\top A)^{-1} A^\top b$, so:

$$\boxed{\hat{b} = A\hat{x} = \underbrace{A(A^\top A)^{-1}A^\top}_{P}\, b}$$

### 3.2  Projection Matrix Properties

The matrix $P = A(A^\top A)^{-1}A^\top$ satisfies:

| Property | Formula | Meaning |
|----------|---------|----------|
| **Idempotence** | $P^2 = P$ | Projecting twice = projecting once |
| **Symmetry** | $P^\top = P$ | $P$ is an orthogonal projector |
| **Complementary** | $(I - P)^2 = I - P$ | $I-P$ projects onto $\mathcal{S}^\perp$ |
| **Eigenvalues** | $\lambda \in \{0, 1\}$ | Vectors in $\mathcal{S}$: $\lambda=1$; in $\mathcal{S}^\perp$: $\lambda=0$ |

### 3.3  Special Case: Orthonormal Columns

If $A = Q$ with $Q^\top Q = I$, then:
$$P = Q(Q^\top Q)^{-1}Q^\top = QQ^\top$$

and the projection coefficients are simply $\hat{x} = Q^\top b$ — no matrix inversion needed.

In [ ]:
# =============================================================================
# Section 3 — Projection Matrix: Implementation & Verification
# =============================================================================

def projection_matrix(A):
    """Compute orthogonal projection matrix onto col(A).

    Formula: P = A (AᵀA)⁻¹ Aᵀ

    Args:
        A: Matrix with full column rank.  Shape: (m, n), n <= m

    Returns:
        P: Projection matrix.  Shape: (m, m)
    """
    return A @ np.linalg.solve(A.T @ A, A.T)

def project_onto(A, b):
    """Project vector b onto col(A).

    Args:
        A: Basis matrix.   Shape: (m, n)
        b: Vector to project.  Shape: (m,)

    Returns:
        b_hat: Projection of b onto col(A).  Shape: (m,)
    """
    P = projection_matrix(A)
    return P @ b

# Test on a 4D example
rng2 = np.random.default_rng(7)
A_test = rng2.standard_normal((6, 2))
b_test = rng2.standard_normal(6)

P_test = projection_matrix(A_test)
b_hat  = P_test @ b_test
err    = b_test - b_hat

print('Projection Matrix Properties:')
check(np.allclose(P_test @ P_test, P_test, atol=ATOL),  'P² = P  (idempotence)')
check(np.allclose(P_test.T, P_test, atol=ATOL),         'Pᵀ = P  (symmetry)')
check(np.allclose(A_test.T @ err, 0, atol=ATOL),        'Aᵀ(b - Pb) = 0  (error ⊥ col(A))')
check(np.linalg.norm(err) < np.linalg.norm(b_test),     '||b - Pb|| < ||b||')

In [ ]:
# =============================================================================
# Section 3 — Geometric Visualization: 2D and 3D Projections
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── 2D: project b onto span{a} ───────────────────────────────────────────────
ax = axes[0]
a2 = np.array([2.0, 1.0])
b2 = np.array([1.0, 2.5])
proj2 = (np.dot(a2, b2) / np.dot(a2, a2)) * a2
err2  = b2 - proj2

origin = np.zeros(2)
def arrow2d(ax, start, vec, color, label):
    ax.annotate('', xy=start+vec, xytext=start,
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    mid = start + vec * 0.55
    ax.text(mid[0], mid[1], label, color=color, fontsize=11, fontweight='bold')

# Extend the line along a
t = np.linspace(-0.2, 1.4, 100)
ax.plot(t * a2[0], t * a2[1], color=GRAY, lw=1, ls='--', label='span{a}')
arrow2d(ax, origin, a2, PRIMARY,    'a')
arrow2d(ax, origin, b2, SECONDARY,  'b')
arrow2d(ax, origin, proj2, TERTIARY,'Pb')
# Right-angle marker
s = 0.08
perp_pt = proj2 + s * (err2 / np.linalg.norm(err2))
corner  = perp_pt + s * (a2 / np.linalg.norm(a2))
ax.plot([proj2[0], perp_pt[0], corner[0]], [proj2[1], perp_pt[1], corner[1]],
        color=GRAY, lw=1)
ax.annotate('', xy=proj2 + err2, xytext=proj2,
            arrowprops=dict(arrowstyle='->', color=ACCENT, lw=1.5, ls='dashed'))
ax.text(proj2[0] + err2[0]*0.5 + 0.1, proj2[1] + err2[1]*0.5,
        'e = b − Pb', color=ACCENT, fontsize=10)
ax.set_xlim(-0.3, 2.8); ax.set_ylim(-0.3, 3.0)
ax.set_aspect('equal'); ax.set_title('2D: Projection onto span{a}', fontsize=13)
ax.legend(loc='upper left', fontsize=10)

# ── 3D: project b onto plane spanned by a1, a2 ───────────────────────────────
ax3 = fig.add_subplot(122, projection='3d')
fig.delaxes(axes[1])   # replace placeholder

a1_3 = np.array([1.0, 0.0, 0.0])
a2_3 = np.array([0.0, 1.0, 0.0])
b3   = np.array([0.7, 0.9, 1.2])
A3   = np.column_stack([a1_3, a2_3])
proj3 = projection_matrix(A3) @ b3
err3  = b3 - proj3

# Draw plane
xx, yy = np.meshgrid(np.linspace(-0.2, 1.2, 4), np.linspace(-0.2, 1.2, 4))
ax3.plot_surface(xx, yy, np.zeros_like(xx), alpha=0.15, color=PRIMARY)

def arrow3d(ax, vec, color, label, offset=(0,0,0.05)):
    ax.quiver(0, 0, 0, vec[0], vec[1], vec[2], color=color, arrow_length_ratio=0.15, lw=2)
    ax.text(vec[0]+offset[0], vec[1]+offset[1], vec[2]+offset[2], label,
            color=color, fontsize=11, fontweight='bold')

arrow3d(ax3, b3,    SECONDARY, 'b',  offset=(0.05, 0, 0.05))
arrow3d(ax3, proj3, TERTIARY,  'Pb', offset=(0.05, 0, -0.1))
ax3.plot([proj3[0], b3[0]], [proj3[1], b3[1]], [proj3[2], b3[2]],
         color=ACCENT, lw=1.5, ls='--')
ax3.text((proj3[0]+b3[0])/2 + 0.05, (proj3[1]+b3[1])/2,
         (proj3[2]+b3[2])/2, 'e', color=ACCENT, fontsize=11)
ax3.set_title('3D: Projection onto a Plane', fontsize=12, pad=10)
ax3.set_xlabel('x'); ax3.set_ylabel('y'); ax3.set_zlabel('z')
ax3.view_init(elev=25, azim=40)

plt.suptitle('Geometric Interpretation of Orthogonal Projection', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 4.  Gram-Schmidt Process

### 4.1  Goal: From Basis to Orthonormal Basis

Given linearly independent vectors $\{a_1, \ldots, a_n\}$, we want to find an **orthonormal basis** $\{q_1, \ldots, q_n\}$ spanning the same space.

### 4.2  Classical Gram-Schmidt (CGS)

At each step $j$, subtract the projections of $a_j$ onto all previously computed basis vectors:

$$v_j = a_j - \sum_{i=1}^{j-1} \langle a_j, q_i \rangle\, q_i, \qquad q_j = \frac{v_j}{\|v_j\|}$$

In matrix form, this is exactly $A = QR$ where:
$$r_{ij} = \langle a_j, q_i \rangle \text{ (for } i < j\text{)}, \quad r_{jj} = \|v_j\|$$

**CGS computes projections using the original vector $a_j$** — fine in exact arithmetic, but in floating point, accumulated rounding errors can cause loss of orthogonality.

### 4.3  Modified Gram-Schmidt (MGS)

MGS projects out each direction one at a time, updating the working vector $v$ after each projection:

$$v_j^{(0)} = a_j$$
$$v_j^{(i)} = v_j^{(i-1)} - \langle v_j^{(i-1)}, q_i \rangle\, q_i, \quad i = 1, \ldots, j-1$$
$$q_j = v_j^{(j-1)} / \| v_j^{(j-1)} \|$$

**Algebraically identical to CGS**, but numerically far superior: each projection step uses the most recently updated residual, preventing error accumulation.

$$\boxed{\text{MGS loss of orthogonality} \sim \epsilon_{\text{mach}} \cdot \kappa(A), \quad \text{CGS} \sim \epsilon_{\text{mach}} \cdot \kappa(A)^2}$$

In [ ]:
# =============================================================================
# Section 4 — Classical Gram-Schmidt (CGS)
# =============================================================================

def gram_schmidt_classical(A):
    """Classical Gram-Schmidt orthogonalization.

    Projects each new vector against ALL previously found basis vectors
    using the original column (before any updates). Numerically unstable
    for ill-conditioned A.

    Args:
        A: Matrix with linearly independent columns.  Shape: (m, n)

    Returns:
        Q: Orthonormal basis matrix.  Shape: (m, n)
        R: Upper-triangular factor.   Shape: (n, n)
    """
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    for j in range(n):
        v = A[:, j].copy()                        # start from original column
        for i in range(j):                        # subtract projections
            R[i, j] = Q[:, i] @ A[:, j]          # <q_i, a_j>  (original a_j!)
            v -= R[i, j] * Q[:, i]
        R[j, j] = np.linalg.norm(v)
        if R[j, j] < 1e-14:
            raise ValueError(f'Column {j} is linearly dependent on previous columns.')
        Q[:, j] = v / R[j, j]

    return Q, R


# Quick test on a well-conditioned matrix
A_demo = np.array([[1, 1, 0],
                   [1, 0, 1],
                   [0, 1, 1]], dtype=float)

Q_cgs, R_cgs = gram_schmidt_classical(A_demo)
print('Classical Gram-Schmidt on well-conditioned matrix:')
check(np.allclose(Q_cgs @ R_cgs, A_demo, atol=ATOL),     'A = QR reconstruction')
check(np.allclose(Q_cgs.T @ Q_cgs, np.eye(3), atol=ATOL),'Qᵀ Q = I  (orthonormality)')
check(np.all(np.diag(R_cgs) > 0),                         'R has positive diagonal')

In [ ]:
# =============================================================================
# Section 4 — Modified Gram-Schmidt (MGS)
# =============================================================================

def gram_schmidt_modified(A):
    """Modified Gram-Schmidt orthogonalization.

    Projects each new vector against ONE direction at a time, updating
    the working residual after each step. Numerically superior to CGS
    for ill-conditioned matrices.

    Args:
        A: Matrix with linearly independent columns.  Shape: (m, n)

    Returns:
        Q: Orthonormal basis matrix.  Shape: (m, n)
        R: Upper-triangular factor.   Shape: (n, n)
    """
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    V = A.copy().astype(float)         # working copy — updated in place

    for i in range(n):
        R[i, i] = np.linalg.norm(V[:, i])
        if R[i, i] < 1e-14:
            raise ValueError(f'Column {i} numerically linearly dependent.')
        Q[:, i] = V[:, i] / R[i, i]
        for j in range(i + 1, n):     # update all subsequent columns
            R[i, j]  = Q[:, i] @ V[:, j]   # <q_i, v_j>  (updated v_j!)
            V[:, j] -= R[i, j] * Q[:, i]   # remove component along q_i

    return Q, R


Q_mgs, R_mgs = gram_schmidt_modified(A_demo)
print('Modified Gram-Schmidt on same matrix:')
check(np.allclose(Q_mgs @ R_mgs, A_demo, atol=ATOL),     'A = QR reconstruction')
check(np.allclose(Q_mgs.T @ Q_mgs, np.eye(3), atol=ATOL),'Qᵀ Q = I  (orthonormality)')
check(np.allclose(Q_mgs, Q_cgs, atol=1e-10),              'CGS and MGS agree on well-conditioned A')

### 4.4  Numerical Instability of CGS: Hilbert Matrix Experiment

The **Hilbert matrix** $H_{ij} = 1/(i+j-1)$ is famously ill-conditioned: $\kappa(H_n) \approx e^{3.5n}$. It exposes the numerical gap between CGS and MGS.

We measure **loss of orthogonality** as $\|Q^\top Q - I\|_F$, which should be $\approx 0$ for an exact computation.

In [ ]:
# =============================================================================
# Section 4 — Numerical Instability: CGS vs MGS on Hilbert Matrix
# =============================================================================

def hilbert_matrix(n):
    """Generate the n×n Hilbert matrix H[i,j] = 1/(i+j+1).

    Args:
        n: Matrix size.

    Returns:
        H: Hilbert matrix.  Shape: (n, n)
    """
    i, j = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
    return 1.0 / (i + j + 1)

sizes = range(3, 16)
cgs_loss, mgs_loss, cond_numbers = [], [], []

for n in sizes:
    H = hilbert_matrix(n)
    cond_numbers.append(np.linalg.cond(H))
    Q_c, _ = gram_schmidt_classical(H)
    Q_m, _ = gram_schmidt_modified(H)
    cgs_loss.append(np.linalg.norm(Q_c.T @ Q_c - np.eye(n), 'fro'))
    mgs_loss.append(np.linalg.norm(Q_m.T @ Q_m - np.eye(n), 'fro'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(list(sizes), cgs_loss, 'o-', color=SECONDARY, label='CGS')
ax.semilogy(list(sizes), mgs_loss, 's-', color=TERTIARY,  label='MGS')
ax.set_xlabel('Matrix size n'); ax.set_ylabel('||QᵀQ − I||_F  (log scale)')
ax.set_title('Loss of Orthogonality: CGS vs MGS\n(Hilbert matrix)', fontsize=12)
ax.legend(); ax.grid(True, which='both', alpha=0.3)
ax.axhline(np.finfo(float).eps, color=GRAY, ls=':', lw=1, label='machine ε')

ax2 = axes[1]
ax2.loglog(cond_numbers, cgs_loss, 'o-', color=SECONDARY, label='CGS  ~ ε κ²')
ax2.loglog(cond_numbers, mgs_loss, 's-', color=TERTIARY,  label='MGS  ~ ε κ')
kappas = np.array(cond_numbers)
eps    = np.finfo(float).eps
ax2.loglog(kappas, eps * kappas,    '--', color=SECONDARY, alpha=0.5, label='ε·κ')
ax2.loglog(kappas, eps * kappas**2, '--', color=TERTIARY,  alpha=0.5, label='ε·κ²')
ax2.set_xlabel('Condition number κ(H)  (log scale)')
ax2.set_ylabel('||QᵀQ − I||_F  (log scale)')
ax2.set_title('Loss of Orthogonality vs Condition Number', fontsize=12)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\nFor n=12 Hilbert matrix:')
n_demo = 12
H12 = hilbert_matrix(n_demo)
Q_c12, _ = gram_schmidt_classical(H12)
Q_m12, _ = gram_schmidt_modified(H12)
print(f'  κ(H₁₂)      = {np.linalg.cond(H12):.2e}')
print(f'  CGS loss    = {np.linalg.norm(Q_c12.T @ Q_c12 - np.eye(n_demo), "fro"):.2e}')
print(f'  MGS loss    = {np.linalg.norm(Q_m12.T @ Q_m12 - np.eye(n_demo), "fro"):.2e}')

---
## 5.  QR Factorization

### 5.1  Definition

For $A \in \mathbb{R}^{m \times n}$ with $m \geq n$ and $\mathrm{rank}(A) = n$, the **thin QR factorization** is:

$$\boxed{A = QR}$$

where $Q \in \mathbb{R}^{m \times n}$ has orthonormal columns ($Q^\top Q = I_n$) and $R \in \mathbb{R}^{n \times n}$ is upper triangular with positive diagonal.

### 5.2  Uniqueness

The thin QR factorization with $r_{ii} > 0$ is **unique**. (Proof: if $A = Q_1 R_1 = Q_2 R_2$, then $Q_2^\top Q_1 = R_2 R_1^{-1}$, which is simultaneously orthogonal and upper triangular — hence diagonal — and with positive diagonal entries must equal $I$.)

### 5.3  Connection to Gram-Schmidt

MGS applied to columns $a_1, \ldots, a_n$ of $A$ produces exactly the QR factorization:

$$a_j = \sum_{i=1}^{j} r_{ij} q_i \quad \Longleftrightarrow \quad A = QR$$

The entry $r_{ij} = \langle q_i, a_j \rangle$ (for $i < j$) records how much of direction $q_i$ was present in $a_j$.

### 5.4  Other QR Algorithms

| Method | Stability | Cost | Notes |
|--------|-----------|------|-------|
| Classical Gram-Schmidt | Poor | $O(mn^2)$ | Pedagogical |
| Modified Gram-Schmidt | Good | $O(mn^2)$ | Recommended for dense $A$ |
| Householder reflections | Excellent | $O(mn^2)$ | LAPACK default (`dgeqrf`) |
| Givens rotations | Excellent | $O(mn^2)$ | Sparse/structured matrices |

In [ ]:
# =============================================================================
# Section 5 — QR Factorization: Implementation & Verification
# =============================================================================

def qr_factorization(A):
    """Compute thin QR factorization via Modified Gram-Schmidt.

    Args:
        A: Full-column-rank matrix.  Shape: (m, n), m >= n

    Returns:
        Q: Orthonormal columns.      Shape: (m, n)
        R: Upper-triangular factor.  Shape: (n, n)
    """
    return gram_schmidt_modified(A)


# Test on a random tall matrix
rng3 = np.random.default_rng(13)
m_t, n_t = 8, 4
A_tall = rng3.standard_normal((m_t, n_t))

Q_t, R_t = qr_factorization(A_tall)

# Compare with numpy.linalg.qr (verification only)
Q_np, R_np = np.linalg.qr(A_tall)
# Signs may differ; align
signs = np.sign(np.diag(R_t)) * np.sign(np.diag(R_np))

print(f'QR Factorization ({m_t}×{n_t} matrix):')
check(np.allclose(Q_t @ R_t, A_tall, atol=ATOL),           'A = QR')
check(np.allclose(Q_t.T @ Q_t, np.eye(n_t), atol=ATOL),   'Qᵀ Q = I')
check(np.allclose(R_t, np.triu(R_t), atol=ATOL),           'R is upper triangular')
check(np.all(np.diag(R_t) > 0),                             'R has positive diagonal')
check(np.allclose(np.abs(R_t), np.abs(R_np), atol=1e-8),   '|R_MGS| matches |R_numpy|  (up to sign)')

print(f'\nR factor (4×4 block):')
print(np.round(R_t, 4))

---
## 6.  Least Squares via QR

### 6.1  Problem Formulation

Given an overdetermined system $Ax = b$ ($m > n$, no exact solution), we seek:
$$\hat{x} = \arg\min_{x \in \mathbb{R}^n} \|Ax - b\|_2^2$$

### 6.2  QR Solution

Substituting $A = QR$ (with $Q^\top Q = I$):
$$\|Ax - b\|^2 = \|QRx - b\|^2 = \|Rx - Q^\top b\|^2 + \|(I - QQ^\top)b\|^2$$

The second term is independent of $x$. The first is minimized by solving the triangular system:

$$\boxed{R\hat{x} = Q^\top b}$$

This is solved by **back-substitution** — no matrix inversion needed.

### 6.3  Why QR is Better than Normal Equations

The normal equations $A^\top A \hat{x} = A^\top b$ form a matrix $A^\top A$ with:
$$\kappa(A^\top A) = \kappa(A)^2$$

Solving via QR works with $A$ directly:
$$\kappa(R) = \kappa(A)$$

For a matrix with $\kappa(A) = 10^6$, normal equations have condition $10^{12}$ — nearly at double-precision limits!

In [ ]:
# =============================================================================
# Section 6 — Least Squares via QR
# =============================================================================

def least_squares_qr(A, b):
    """Solve least-squares problem min ||Ax - b|| via QR factorization.

    Uses A = QR  →  R x = Qᵀ b  (solved by back-substitution).

    Args:
        A: Design matrix.   Shape: (m, n), m >= n, full column rank
        b: Right-hand side. Shape: (m,)

    Returns:
        x_hat: Least-squares solution.  Shape: (n,)
        residual_norm: ||A x_hat - b||. Scalar.
    """
    Q, R = qr_factorization(A)
    Qtb  = Q.T @ b                     # project b onto col(A)
    x_hat = np.linalg.solve(R, Qtb)    # back-substitution (R is upper triangular)
    residual_norm = np.linalg.norm(A @ x_hat - b)
    return x_hat, residual_norm


# Example: fit a quadratic to noisy data
rng4 = np.random.default_rng(99)
t_data  = np.linspace(0, 3, 20)
y_true  = 2.0 * t_data**2 - 1.5 * t_data + 0.5
y_noisy = y_true + rng4.standard_normal(20) * 0.3

# Design matrix for polynomial degree 2
A_poly = np.column_stack([np.ones_like(t_data), t_data, t_data**2])

x_qr, res_qr = least_squares_qr(A_poly, y_noisy)
x_np = np.linalg.lstsq(A_poly, y_noisy, rcond=None)[0]   # numpy reference

print('Least Squares via QR (quadratic fit):')
print(f'  True coeffs  : [0.5, -1.5, 2.0]')
print(f'  QR solution  : [{x_qr[0]:.4f}, {x_qr[1]:.4f}, {x_qr[2]:.4f}]')
print(f'  numpy lstsq  : [{x_np[0]:.4f}, {x_np[1]:.4f}, {x_np[2]:.4f}]')
print(f'  Residual norm: {res_qr:.4f}')
check(np.allclose(x_qr, x_np, atol=1e-8), 'QR solution matches numpy.linalg.lstsq')

# Plot
t_fine = np.linspace(0, 3, 200)
y_fit  = x_qr[0] + x_qr[1]*t_fine + x_qr[2]*t_fine**2

plt.figure(figsize=(10, 4))
plt.scatter(t_data, y_noisy, color=SECONDARY, s=40, zorder=5, label='Noisy data')
plt.plot(t_fine, y_fit, color=PRIMARY, lw=2.5, label='QR least-squares fit (degree 2)')
plt.plot(t_fine, 2*t_fine**2 - 1.5*t_fine + 0.5, '--', color=GRAY, lw=1.5, label='True quadratic')
plt.xlabel('t'); plt.ylabel('y')
plt.title('Least Squares via QR: Quadratic Curve Fitting')
plt.legend(); plt.tight_layout(); plt.show()

---
## 7.  Least Squares via Normal Equations

### 7.1  The Normal Equations

Setting the gradient of $\|Ax - b\|^2$ to zero:
$$\frac{\partial}{\partial x} \|Ax - b\|^2 = 2A^\top(Ax - b) = 0$$

$$\boxed{A^\top A\, \hat{x} = A^\top b}$$

When $A$ has full column rank, $A^\top A$ is symmetric positive definite and the solution is unique:
$$\hat{x} = (A^\top A)^{-1} A^\top b$$

### 7.2  Cholesky Factorization

Since $A^\top A \succ 0$, we can use the **Cholesky factorization** $A^\top A = LL^\top$ and solve:
$$L y = A^\top b, \quad L^\top \hat{x} = y$$

This is faster than general LU for symmetric PD matrices, but still operates on the squared condition number.

### 7.3  When Normal Equations Fail

For ill-conditioned $A$ with $\kappa(A) \approx \sqrt{1/\varepsilon_{\text{mach}}} \approx 10^8$:
- $\kappa(A^\top A) \approx 10^{16}$ — at the edge of double precision
- $A^\top A$ may not even be computed accurately due to **catastrophic cancellation**
- QR avoids forming $A^\top A$ entirely

In [ ]:
# =============================================================================
# Section 7 — Least Squares via Normal Equations & Condition Number Comparison
# =============================================================================

def least_squares_normal_equations(A, b):
    """Solve least-squares problem via normal equations A'A x = A'b.

    Uses Cholesky factorization of A'A for efficiency.

    Args:
        A: Design matrix.   Shape: (m, n), m >= n, full column rank
        b: Right-hand side. Shape: (m,)

    Returns:
        x_hat: Least-squares solution.  Shape: (n,)
        residual_norm: ||A x_hat - b||. Scalar.
    """
    AtA = A.T @ A
    Atb = A.T @ b
    # Cholesky: AtA = L L'
    L = np.linalg.cholesky(AtA)
    # Forward substitution: L y = A'b
    y = np.linalg.solve(L, Atb)
    # Back substitution: L' x = y
    x_hat = np.linalg.solve(L.T, y)
    residual_norm = np.linalg.norm(A @ x_hat - b)
    return x_hat, residual_norm


# Verify normal equations give same answer as QR on well-conditioned problem
x_ne, res_ne = least_squares_normal_equations(A_poly, y_noisy)
print('Normal Equations Solution (same quadratic fit):')
print(f'  NE  solution : [{x_ne[0]:.4f}, {x_ne[1]:.4f}, {x_ne[2]:.4f}]')
print(f'  QR  solution : [{x_qr[0]:.4f}, {x_qr[1]:.4f}, {x_qr[2]:.4f}]')
check(np.allclose(x_ne, x_qr, atol=1e-8), 'NE and QR agree on well-conditioned problem')

print()

# ── Condition Number Comparison: Ill-conditioned Vandermonde system ───────────
print('Condition Number Comparison (Vandermonde matrix, increasing degree):')
print(f'  {"degree":>6}  {"κ(A)":>12}  {"κ(AᵀA)":>14}  {"||x_QR - x_NE||":>18}')
print('  ' + '-'*56)

t_vand = np.linspace(0, 1, 20)
y_vand = np.sin(2 * np.pi * t_vand) + 0.05 * rng4.standard_normal(20)

for deg in [3, 5, 7, 9, 11, 13]:
    A_v = np.vander(t_vand, deg + 1, increasing=True)
    kA   = np.linalg.cond(A_v)
    kAtA = np.linalg.cond(A_v.T @ A_v)
    x_q, _ = least_squares_qr(A_v, y_vand)
    try:
        x_n, _ = least_squares_normal_equations(A_v, y_vand)
        diff = np.linalg.norm(x_q - x_n)
    except np.linalg.LinAlgError:
        diff = float('nan')
    print(f'  {deg:>6}  {kA:>12.2e}  {kAtA:>14.2e}  {diff:>18.2e}')

In [ ]:
# =============================================================================
# Section 7 — Visual Comparison: QR vs Normal Equations on Ill-Conditioned Problem
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

t_plot = np.linspace(0, 1, 200)
degrees_to_show = [5, 9, 13]

for ax, deg in zip(axes, degrees_to_show):
    A_v = np.vander(t_vand, deg + 1, increasing=True)
    A_p = np.vander(t_plot, deg + 1, increasing=True)

    x_q, _ = least_squares_qr(A_v, y_vand)
    y_q = A_p @ x_q

    try:
        x_n, _ = least_squares_normal_equations(A_v, y_vand)
        y_n = A_p @ x_n
        ne_ok = True
    except Exception:
        ne_ok = False

    ax.scatter(t_vand, y_vand, s=20, color=GRAY, zorder=5, label='data')
    ax.plot(t_plot, np.sin(2*np.pi*t_plot), '--', color=GRAY, lw=1, label='true')
    ax.plot(t_plot, np.clip(y_q, -2, 2), color=PRIMARY, lw=2, label='QR')
    if ne_ok:
        ax.plot(t_plot, np.clip(y_n, -2, 2), color=SECONDARY, lw=1.5, ls='--', label='NE')
    ax.set_ylim(-2, 2)
    kappa = np.linalg.cond(A_v)
    ax.set_title(f'Degree {deg}\nκ(A) = {kappa:.1e}', fontsize=11)
    ax.set_xlabel('t'); ax.legend(fontsize=8)

plt.suptitle('QR vs Normal Equations: Vandermonde Polynomial Fitting', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 8.  Function Approximation in $L^2[-\pi, \pi]$

### 8.1  Orthogonal Projection in Function Spaces

The same projection theory applies in infinite-dimensional function spaces. With inner product
$$\langle f, g \rangle = \int_{-\pi}^{\pi} f(x)\, g(x)\, dx$$

we can project any $f \in L^2[-\pi,\pi]$ onto a finite-dimensional subspace $\mathcal{P}_n = \mathrm{span}\{1, x, x^2, \ldots, x^n\}$.

### 8.2  Polynomial Orthonormal Basis via Gram-Schmidt

The monomials $\{1, x, x^2, \ldots\}$ are linearly independent in $L^2[-\pi,\pi]$ but **not orthogonal**. We apply Gram-Schmidt to obtain an orthonormal polynomial basis $\{p_0, p_1, \ldots, p_n\}$:

1. Discretize $[-\pi, \pi]$ with $N$ quadrature points
2. Evaluate monomials at quadrature points → matrix $V \in \mathbb{R}^{N \times (n+1)}$
3. Compute $V = QR$ (MGS) — $Q$ columns are the discretized orthonormal polynomials
4. Project $f(x) = \sin(x)$: coefficients $c = Q^\top f$, approximation $\hat{f} = Qc = QQ^\top f$

### 8.3  Convergence

The projection error satisfies:
$$\|f - P_n f\|_{L^2} = \|f\|^2 - \sum_{k=0}^{n} |\langle f, p_k\rangle|^2 \xrightarrow{n\to\infty} 0$$

This follows from the **Parseval identity** and the completeness of polynomial bases in $L^2[-\pi,\pi]$.

Note: $\sin(x)$ is an **odd function**, so its projection onto even-degree polynomials is zero. The approximation improves through the odd-degree terms only.

In [ ]:
# =============================================================================
# Section 8 — Function Approximation: sin(x) via Orthogonal Projection
# =============================================================================

def l2_inner_product(f, g, x, w=None):
    """Approximate L² inner product via quadrature.

    <f, g> ≈ sum_i w_i f(x_i) g(x_i)

    Args:
        f: Function values at quadrature points.  Shape: (N,)
        g: Function values at quadrature points.  Shape: (N,)
        x: Quadrature nodes.                       Shape: (N,)
        w: Quadrature weights. If None, use uniform (trapezoid rule). Shape: (N,)

    Returns:
        Approximate inner product (scalar).
    """
    if w is None:
        dx = x[1] - x[0]   # uniform spacing assumed
        return np.trapz(f * g, x)
    return float(w @ (f * g))


def polynomial_approximation(f_vals, x, degree):
    """Approximate f via orthogonal projection onto polynomial space P_degree.

    Builds Vandermonde matrix, orthogonalizes via MGS (scaled for L² norm),
    then projects f.

    Args:
        f_vals: Function values at quadrature points.  Shape: (N,)
        x:      Quadrature nodes in [-pi, pi].         Shape: (N,)
        degree: Polynomial degree for approximation.

    Returns:
        f_approx: Approximation values at x.           Shape: (N,)
        coeffs:   Projection coefficients.             Shape: (degree+1,)
        Q:        Orthonormal polynomial basis matrix. Shape: (N, degree+1)
        l2_error: ||f - f_approx||_L2 (scalar).
    """
    N = len(x)
    dx = x[1] - x[0]

    # Build Vandermonde (monomials scaled so L² norm ~ O(1))
    x_scaled = x / np.pi   # normalize to [-1, 1]
    V = np.column_stack([x_scaled**k for k in range(degree + 1)])  # shape (N, d+1)

    # Weight matrix for discrete L² inner product: <f,g>_h = dx * f'g
    # Incorporate sqrt(dx) into V so that standard Euclidean inner product
    # on the scaled matrix = discrete L² inner product
    V_scaled = V * np.sqrt(dx)   # shape (N, d+1)

    Q_scaled, R = gram_schmidt_modified(V_scaled)
    Q = Q_scaled / np.sqrt(dx)   # restore original scaling; Q columns orthonormal in L²

    # Project: coefficients c_k = <f, q_k>_L2 ≈ dx * f' q_k
    coeffs   = (Q * dx).T @ f_vals   # shape (d+1,)
    f_approx = Q @ coeffs            # shape (N,)

    l2_error = np.sqrt(np.trapz((f_vals - f_approx)**2, x))
    return f_approx, coeffs, Q, l2_error


# ── Set up quadrature grid ────────────────────────────────────────────────────
N_quad = 500
x_grid = np.linspace(-np.pi, np.pi, N_quad)
f_sin  = np.sin(x_grid)

# ── Approximate sin(x) at various polynomial degrees ─────────────────────────
degrees = [1, 3, 5, 7, 9, 11]
errors  = []

print('L² Approximation of sin(x) on [-π, π]:')
print(f'  {"Degree":>6}  {"L² error":>12}  {"Max error":>12}')
print('  ' + '-'*34)

for deg in degrees:
    f_hat, c, Q_basis, err_l2 = polynomial_approximation(f_sin, x_grid, deg)
    err_inf = np.max(np.abs(f_sin - f_hat))
    errors.append(err_l2)
    print(f'  {deg:>6}  {err_l2:>12.4e}  {err_inf:>12.4e}')

In [ ]:
# =============================================================================
# Section 8 — Visualization: Function Approximation
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ── Left: Approximations at selected degrees ──────────────────────────────────
ax = axes[0]
ax.plot(x_grid, f_sin, 'k-', lw=2.5, label='sin(x)', zorder=10)
colors_deg = [PRIMARY, SECONDARY, TERTIARY, ACCENT, 'darkorange', 'brown']
show_degrees = [1, 3, 5, 7, 11]

for i, deg in enumerate(show_degrees):
    f_hat, _, _, _ = polynomial_approximation(f_sin, x_grid, deg)
    lw = 1.5 + 0.3 * i
    ax.plot(x_grid, f_hat, color=colors_deg[i], lw=lw,
            ls='--' if deg < 7 else '-',
            label=f'Degree {deg}', alpha=0.85)

ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Polynomial Approximations of sin(x)\n'
             'via Orthogonal Projection in L²[-π, π]', fontsize=11)
ax.legend(fontsize=9); ax.set_xlim(-np.pi, np.pi)
ax.set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax.set_xticklabels(['-π', '-π/2', '0', 'π/2', 'π'])

# ── Right: L² error vs degree ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.semilogy(degrees, errors, 'o-', color=PRIMARY, lw=2, ms=8)
ax2.set_xlabel('Polynomial degree'); ax2.set_ylabel('L² error  (log scale)')
ax2.set_title('Convergence of L² Approximation Error', fontsize=11)
ax2.set_xticks(degrees)
ax2.grid(True, which='both', alpha=0.3)

for i, (d, e) in enumerate(zip(degrees, errors)):
    ax2.annotate(f'{e:.1e}', (d, e), textcoords='offset points',
                 xytext=(6, 4), fontsize=8, color=PRIMARY)

plt.suptitle('Function Approximation via Orthogonal Projection', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Section 8 — Orthonormal Polynomial Basis Visualization
# =============================================================================

# Show the first 6 orthonormal polynomial basis functions
deg_basis = 5
_, _, Q_show, _ = polynomial_approximation(f_sin, x_grid, deg_basis)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for k in range(deg_basis + 1):
    ax = axes[k]
    ax.plot(x_grid, Q_show[:, k], color=colors_deg[k], lw=2)
    ax.axhline(0, color=GRAY, lw=0.8, ls='--')
    ax.set_title(f'$p_{k}(x)$  (degree {k})', fontsize=11)
    ax.set_xlim(-np.pi, np.pi)
    ax.set_xticks([-np.pi, 0, np.pi])
    ax.set_xticklabels(['-π', '0', 'π'])
    # Annotate L² norm
    norm_k = np.sqrt(np.trapz(Q_show[:, k]**2, x_grid))
    ax.text(0.97, 0.97, f'||p_{k}|| = {norm_k:.4f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color=GRAY)

plt.suptitle('Orthonormal Polynomial Basis for $L^2[-\\pi, \\pi]$\n'
             '(via Gram-Schmidt on scaled monomials)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Verify orthonormality of the discretized basis
dx = x_grid[1] - x_grid[0]
GramMatrix = dx * (Q_show.T @ Q_show)   # discrete Gram matrix
orth_err = np.linalg.norm(GramMatrix - np.eye(deg_basis + 1), 'fro')
check(orth_err < 1e-6, f'Gram matrix ||G - I||_F = {orth_err:.2e}  (orthonormality in L²)')

---
## 9.  Summary & References

### 9.1  Key Results Summary

| Topic | Key Formula | Notes |
|-------|------------|-------|
| Orthogonal projection | $P = A(A^\top A)^{-1}A^\top$ | Idempotent, symmetric |
| ONB case | $P = QQ^\top$ | No inversion needed |
| CGS | $r_{ij} = \langle q_i, a_j\rangle$, subtract then normalize | Numerically unstable |
| MGS | Same formulas, update residual after each projection | $O(\varepsilon_{\text{mach}}\kappa)$ orthogonality loss |
| QR factorization | $A = QR$, $Q^\top Q = I$, $R$ upper triangular | Unique with $r_{ii} > 0$ |
| Least squares (QR) | $R\hat{x} = Q^\top b$ | Condition $\kappa(A)$ |
| Least squares (NE) | $A^\top A\hat{x} = A^\top b$ | Condition $\kappa(A)^2$ |
| Function approx | $\hat{f} = QQ^\top f$ (discrete $L^2$) | Converges as degree $\to \infty$ |

### 9.2  Key Takeaways

1. **Orthogonality is universal**: The same projection formula works in $\mathbb{R}^n$, weighted Euclidean spaces, and $L^2$ function spaces — only the inner product changes.

2. **Modified Gram-Schmidt over Classical**: In exact arithmetic they are identical; in floating point, MGS loses orthogonality proportional to $\kappa(A)$ while CGS loses it proportional to $\kappa(A)^2$. Always prefer MGS.

3. **QR over Normal Equations**: Forming $A^\top A$ squares the condition number. For problems where $\kappa(A) > 10^4$, the normal equations approach accumulates significant numerical error. QR (especially via Householder) is the standard in production software (LAPACK `dgels`).

4. **Function approximation as projection**: Polynomial regression is exactly orthogonal projection in a discretized $L^2$ space. Building an orthonormal polynomial basis first (via MGS/QR on the Vandermonde matrix) makes the projection coefficients easy to compute and the basis well-conditioned.

5. **Parity matters**: Since $\sin(x)$ is odd, only odd-degree polynomial basis functions contribute to its approximation — the even-degree projection coefficients are exactly zero.

### 9.3  References

- **Trefethen, L. N., & Bau, D.** (1997). *Numerical Linear Algebra*. SIAM. — Lectures 7–11 (QR), 18–19 (Least squares)
- **Golub, G. H., & Van Loan, C. F.** (2013). *Matrix Computations* (4th ed.). Johns Hopkins. — Chapter 5 (Orthogonalization and Least Squares)
- **Björck, Å.** (1994). Numerics of Gram-Schmidt orthogonalization. *Linear Algebra and its Applications*, 197–198, 297–316.
- **Strang, G.** (2016). *Introduction to Linear Algebra* (5th ed.). Wellesley-Cambridge Press. — Chapter 4 (Orthogonality)
- **Higham, N. J.** (2002). *Accuracy and Stability of Numerical Algorithms* (2nd ed.). SIAM. — Chapter 19 (QR factorization)